In [4]:
import numpy as np

In [5]:
vec = [1,3,2,4,3]
np.sort(vec)

array([1, 2, 3, 3, 4])

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, LeaveOneOut
import matplotlib.pyplot as plt

from scipy import stats
from sklearn.metrics import f1_score, roc_auc_score, auc, roc_curve, precision_score, recall_score, confusion_matrix, accuracy_score
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_validate


%matplotlib inline 

In [16]:
pd.set_option("display.max_rows",25000)

In [3]:
df_classificacao = pd.read_csv('Bases/classificacao_1.csv')

In [2]:
def results_holdout(clf, X_train, y_train, X_test, y_test):
    
    classifier = clf.fit(X_train, np.ravel(y_train))
    metrics = [f1_score, precision_score, recall_score, accuracy_score, roc_auc_score]
    
    pred_train = classifier.predict(X_train)
    pred_test = classifier.predict(X_test)

    prob_train = classifier.predict_proba(X_train)
    prob_train_true = pd.DataFrame(prob_train).iloc[:,1] #prob de ser 1
    prob_test = classifier.predict_proba(X_test)
    prob_test_true = pd.DataFrame(prob_test).iloc[:,1]
    
    results = pd.DataFrame()
    
    for m in metrics:
        if m == roc_auc_score:
                result_train = m(y_train, prob_train_true)
                result_test = m(y_test, prob_test_true)
        else:        
            result_train = m(y_train, pred_train)
            result_test = m(y_test, pred_test)
        
        dict_res = {'Treino': [result_train], 'Teste': [result_test]}
        res_aux = pd.DataFrame(dict_res, index = [str(m.__name__)])
        
        results = pd.concat([results, res_aux], axis = 0)
    
    
    train_res = pd.concat([y_train, pd.DataFrame(prob_train).iloc[:,1]], axis = 1)
    test_res = pd.concat([y_test, pd.DataFrame(prob_test).iloc[:,1]], axis = 1)
    train_res.columns = ['y_train', 'prob']
    test_res.columns = ['y_test', 'prob']
    
    train_prob_True1 = train_res[train_res.y_train == 1]
    train_prob_True0 = train_res[train_res.y_train == 0]
    
    test_prob_True1 = test_res[test_res.y_test == 1]
    test_prob_True0 = test_res[test_res.y_test == 0]
    
    dict_res_ks = {'Treino': [stats.ks_2samp(train_prob_True1.prob, train_prob_True0.prob).statistic], 
                   'Teste': [stats.ks_2samp(test_prob_True1.prob, test_prob_True0.prob).statistic]}
    
    res_aux_ks = pd.DataFrame(dict_res_ks, index = ['KS'])
    results = pd.concat([results, res_aux_ks], axis = 0)
    
    return results

def results_cv(clf, X_train, y_train, kcv):
    
    classifier = clf.fit(X_train, np.ravel(y_train))
    metrics = ['f1', 'precision', 'recall', 'accuracy', 'roc_auc']
    
    results = pd.DataFrame()
    for m in metrics:
        cv_results = cross_validate(clf, X_train, np.ravel(y_train), scoring = m, cv = kcv, return_train_score = True)
        dict_res = {'' + m + '_Treino': cv_results['train_score'], 
                    '' + m + '_Teste': cv_results['test_score']}
        res_aux = pd.DataFrame(dict_res)
        results = pd.concat([results, res_aux], axis = 1)
        
    results_mean = np.transpose(pd.DataFrame(results.mean(), columns=['media']))
    results = pd.concat([results, results_mean], axis = 0)
    return results


def grid_search(clf, X_train, y_train, params, score, cv):    
    grid = GridSearchCV(clf, params, scoring = score, cv = cv, return_train_score=True)
    grid_fitted = grid.fit(X_train, np.ravel(y_train))
    print ("Best score: %.4f" % grid_fitted.best_score_)
    print ("Best parameters: %s" % grid_fitted.best_params_)
    return grid_fitted, grid_fitted.best_estimator_, grid_fitted.cv_results_

In [5]:
def data_info(data):        
    info = pd.DataFrame()
    info['var'] = data.columns
    info['# missing'] = list(data.isnull().sum())
    info['% missing'] = info['# missing'] / data.shape[0]
    info['types'] = list(data.dtypes)
    info['unique values'] = list(len(data[var].unique()) for var in data.columns)
    
    return info

In [26]:
from sklearn.model_selection import StratifiedKFold, KFold, LeaveOneOut

cv_kfold = KFold(5, shuffle = False, random_state=12) #### se shuffle = False os k folds são sequenciais
cv_leave_one_out = LeaveOneOut()

In [11]:
data_info(df_classificacao)

,var,# missing,% missing,types,unique values
0,v_1,0,0.0,float64,1250
1,v_2,0,0.0,float64,1247
2,v_3,0,0.0,float64,1250
3,v_4,0,0.0,float64,1247
4,v_5,0,0.0,float64,1250
5,v_6,0,0.0,float64,1250
6,v_7,0,0.0,float64,1249
7,v_8,0,0.0,float64,1250
8,v_9,0,0.0,float64,1250
9,v_10,0,0.0,float64,1248


SVM

In [8]:
from sklearn.svm import SVC
svc = SVC(C=1.0, kernel='linear')

In [14]:
results_cv(svc, df_classificacao.drop(['target'],axis=1), df_classificacao['target'], 10)

,f1_Treino,f1_Teste,precision_Treino,precision_Teste,recall_Treino,recall_Teste,accuracy_Treino,accuracy_Teste,roc_auc_Treino,roc_auc_Teste
0,0.825260,0.756303,0.836842,0.849057,0.813993,0.681818,0.820285,0.769841,0.881180,0.865657
1,0.825151,0.800000,0.833043,0.847458,0.817406,0.757576,0.819395,0.801587,0.881114,0.865404
2,0.820380,0.818898,0.831874,0.838710,0.809199,0.800000,0.815111,0.816000,0.880275,0.868974
3,0.828720,0.765625,0.841828,0.777778,0.816014,0.753846,0.824000,0.760000,0.882830,0.848718
4,0.819672,0.843750,0.830420,0.857143,0.809199,0.830769,0.814222,0.840000,0.880297,0.871538
5,0.823123,0.820896,0.833916,0.797101,0.812606,0.846154,0.817778,0.808000,0.884154,0.840769
6,0.826575,0.808824,0.837413,0.774648,0.816014,0.846154,0.821333,0.792000,0.882504,0.856154
7,0.819672,0.875000,0.830420,0.888889,0.809199,0.861538,0.814222,0.872000,0.873600,0.929231
8,0.818024,0.833333,0.832451,0.820896,0.804089,0.846154,0.813499,0.822581,0.877295,0.904824
9,0.817786,0.800000,0.837500,0.833333,0.798978,0.769231,0.814387,0.798387,0.879703,0.875880


Naive Bayes

In [2]:
df_classificacao_2 = pd.read_csv('Bases/classificacao_2.csv')

In [23]:
from sklearn.naive_bayes import GaussianNB, BernoulliNB, MultinomialNB
nb = GaussianNB()
results_cv(nb, df_classificacao_2.drop(['target'],axis=1), df_classificacao_2['target'], 5)

,f1_Treino,f1_Teste,precision_Treino,precision_Teste,recall_Treino,recall_Teste,accuracy_Treino,accuracy_Teste,roc_auc_Treino,roc_auc_Teste
0,0.656716,0.646048,0.720617,0.696296,0.603226,0.602564,0.804402,0.794411,0.869133,0.848662
1,0.642984,0.583026,0.716832,0.681034,0.582931,0.509677,0.799000,0.774000,0.853977,0.851669
2,0.645276,0.633803,0.722555,0.697674,0.582931,0.580645,0.801000,0.792000,0.864625,0.834483
3,0.647111,0.615942,0.722222,0.702479,0.586151,0.548387,0.801500,0.788000,0.861424,0.854474
4,0.626786,0.710345,0.703407,0.762963,0.565217,0.664516,0.791104,0.831663,0.852340,0.889141
media,0.643775,0.637833,0.717126,0.708089,0.584091,0.581158,0.799401,0.796015,0.860300,0.855686


## Prova ITAu


In [2]:
df_classificacao_2 = pd.read_csv('Bases/classificacao_2.csv')

In [6]:
data_info(df_classificacao_2)

,var,# missing,% missing,types,unique values
0,v_1,0,0.0,float64,2492
1,v_2,0,0.0,float64,2498
2,v_3,0,0.0,float64,2496
3,v_4,0,0.0,float64,2498
4,v_5,0,0.0,float64,2495
5,v_6,0,0.0,float64,2496
6,v_7,0,0.0,float64,2494
7,v_8,0,0.0,float64,2494
8,v_9,0,0.0,float64,2494
9,v_10,0,0.0,float64,2496


In [23]:
X_train = df_classificacao_2.drop(['target'],axis=1).loc[0:2000,:]
y_train = df_classificacao_2['target'].loc[0:2000]
X_test =  df_classificacao_2.drop(['target'],axis=1).loc[2001:,:]
y_test =  df_classificacao_2['target'].loc[2001:]

In [25]:
X_test.shape

(499, 12)

In [3]:
from sklearn.naive_bayes import GaussianNB, BernoulliNB, MultinomialNB
nb = GaussianNB()
results_holdout(nb, X_train, y_train, X_test, y_test)

NameError: name 'results_holdout' is not defined

## Kfold sequencial por porcentagem

In [35]:
from sklearn.metrics import accuracy_score
X_test = pd.DataFrame()
X_train = pd.DataFrame()
y_test = pd.DataFrame()
y_train = pd.DataFrame()
acurracy_vec = [] 
nb = GaussianNB()

for i in range(2,3):
    X_test = pd.DataFrame()
    X_train = pd.DataFrame()
    y_test = pd.DataFrame()
    y_train = pd.DataFrame()
    for j in range(0,10):
        if i==j:
            X_test =   pd.DataFrame(df_classificacao_2.drop(['target'],axis=1).loc[np.arange(250*j,250*(j+1))])
            y_test =  pd.DataFrame(df_classificacao_2['target'].loc[np.arange(250*j,250*(j+1))])
        else:
            X_train = X_train.append(pd.DataFrame(df_classificacao_2.drop(['target'],axis=1).loc[np.arange(250*j,250*(j+1))]))
            y_train = y_train.append(pd.DataFrame(df_classificacao_2['target'].loc[np.arange(250*j,250*(j+1))]))
    nb.fit(X_train,y_train)
    y_pred= nb.predict(X_train)
    acurracy_vec.append(accuracy_score(y_train, y_pred))
print(round(np.mean(acurracy_vec),3))

0.802


C:\Users\diego\Anaconda3\lib\site-packages\sklearn\utils\validation.py:578: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


## LeaveOneOut

In [38]:
loo.get_n_splits(X)


2500

In [13]:
import numpy as np
from sklearn.model_selection import LeaveOneOut
from sklearn.naive_bayes import GaussianNB, BernoulliNB, MultinomialNB
X = df_classificacao_2.drop(['target'],axis=1)
y = df_classificacao_2['target']
loo = LeaveOneOut()
loo.get_n_splits(X)
acurracy_vec_train = []
acurracy_vec_test = []
nb = GaussianNB()


for train_index, test_index in loo.split(X):
    #print("TRAIN:", train_index, "TEST:", test_index)
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    #print(X_train, X_test, y_train, y_test)
    nb.fit(X_train,y_train)
    y_pred_train= nb.predict(X_train)
    y_pred_test= nb.predict(X_test)
    acurracy_vec_train.append(accuracy_score(y_train, y_pred_train))
    acurracy_vec_test.append(accuracy_score(y_test, y_pred_test))
#print(acurracy_vec)
print('Média base treino: ',round(np.mean(acurracy_vec_train),3))
print('Média base test: ',round(np.mean(acurracy_vec_test),3))


Média base treino:  0.799
Média base test:  0.797
